#### DISTRIBUTION PROFILE

##### 04.1 DOCUMENTAÇÃO DO DISTRIBUTION PROFILE

###### Objetivo

O *Distribution Profile* tem como objetivo analisar como os valores estão distribuídos dentro de cada coluna do DataFrame.

Enquanto o *Schema Profile* avalia as características estruturais das colunas, o *Distribution Profile* analisa o comportamento e a distribuição dos valores observados.

A análise considera automaticamente as colunas elegíveis para distribuição, sem depender do significado de negócio dos atributos.

###### Principais análises

- Resumo estatístico da distribuição;
- Frequência dos valores;
- Principais valores por coluna;
- Concentração dos valores;
- Índice de diversidade;
- Índice de dominância;
- Regularidade da distribuição;
- Identificação de distribuições altamente concentradas;
- Identificação de distribuições altamente diversificadas;
- Consolidação das métricas de distribuição.

###### Característica

O *Distribution Profile* é independente da origem dos dados.

Sua análise é realizada exclusivamente sobre o DataFrame disponibilizado pelo *Data Preparation*.

Dessa forma, o mesmo Profile pode ser utilizado em diferentes fontes e estruturas de dados.

###### Resultado esperado

Ao final desta etapa estarão disponíveis:

- métricas de distribuição por coluna;
- frequência dos valores;
- principais valores observados;
- concentração Top 1, Top 3 e Top 5;
- índice de diversidade;
- índice de dominância;
- indicadores de regularidade;
- DataFrame consolidado do Distribution Profile;
- visualizações para análise da distribuição dos dados.

In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 04.3 PARÂMETROS DO PROFILE
# ============================================================

# O QUE FAZ:
# Define os parâmetros específicos utilizados pelas análises do Distribution Profile.

# COMO FAZ:
# Define a quantidade de valores exibidos nas análises Top N e os níveis utilizados para medir a concentração dos valores.

# POR QUE É IMPORTANTE:
# Mantém as configurações específicas do Distribution Profile próximas às análises que utilizam esses parâmetros, sem sobrecarregar o notebook central de bibliotecas.

# PERGUNTA RESPONDIDA:
# "Quais parâmetros serão utilizados nas análises de distribuição?"

TOP_N = 5

TOP_CONCENTRATION_N = [1, 3, 5]

print(f"TOP_N: {TOP_N}")
print(f"TOP_CONCENTRATION_N: {TOP_CONCENTRATION_N}")

In [0]:
# ============================================================
# 04.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame preparado e as informações necessárias para o Distribution Profile estão disponíveis.

# COMO FAZ:
# Verifica a existência do DataFrame preparado, do total de registros e da lista de colunas elegíveis para análise de distribuição.

# POR QUE É IMPORTANTE:
# Garante que o Profile seja executado sobre uma estrutura válida e evita processamento desnecessário.

# PERGUNTA RESPONDIDA:
# "O DataFrame possui as informações necessárias para executar o Distribution Profile?"

if "df_prepared" not in locals():
    raise ValueError(
        "O DataFrame 'df_prepared' não foi disponibilizado pelo Data Preparation."
    )

if "total_registros" not in locals():
    raise ValueError(
        "A variável 'total_registros' não foi disponibilizada pelo Data Preparation."
    )

if "colunas_distribution" not in locals():
    raise ValueError(
        "A variável 'colunas_distribution' não foi disponibilizada pelo Data Preparation."
    )

if total_registros == 0:
    raise ValueError(
        "O DataFrame preparado não possui registros."
    )

if len(colunas_distribution) == 0:
    raise ValueError(
        "Não existem colunas elegíveis para o Distribution Profile."
    )

print("DataFrame validado com sucesso.")
print(f"Total de registros: {total_registros}")
print(f"Colunas analisáveis: {len(colunas_distribution)}")

In [0]:
# ============================================================
# 04.5 RESUMO DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Calcula um resumo estatístico das colunas elegíveis para o Distribution Profile.

# COMO FAZ:
# Utiliza as funções estatísticas nativas do Spark para calcular quantidade de valores preenchidos, média, desvio padrão, mínimo e máximo para cada coluna numérica.

# POR QUE É IMPORTANTE:
# Fornece uma visão inicial do comportamento quantitativo dos dados e cria uma base para análises posteriores.

# PERGUNTA RESPONDIDA:
# "Como os valores numéricos estão distribuídos em cada coluna?"

metricas_distribuicao = []

for coluna in colunas_distribution:

    campo = next(
        campo
        for campo in df_prepared.schema.fields
        if campo.name == coluna
    )

    coluna_ref = F.col(coluna)

    if isinstance(
        campo.dataType,
        (
            # Tipos numéricos
            __import__("pyspark").sql.types.NumericType
        )
    ):
        metricas_distribuicao.extend([
            F.count(coluna_ref).alias(f"{coluna}__preenchidos"),
            F.avg(coluna_ref).alias(f"{coluna}__media"),
            F.stddev(coluna_ref).alias(f"{coluna}__desvio_padrao"),
            F.min(coluna_ref).alias(f"{coluna}__minimo"),
            F.max(coluna_ref).alias(f"{coluna}__maximo")
        ])

resumo_distribuicao_row = (
    df_prepared
    .agg(*metricas_distribuicao)
    .first()
)

resumo_distribuicao = []

for coluna in colunas_distribution:

    campo = next(
        campo
        for campo in df_prepared.schema.fields
        if campo.name == coluna
    )

    if isinstance(
        campo.dataType,
        __import__("pyspark").sql.types.NumericType
    ):

        resumo_distribuicao.append(
            (
                coluna,
                resumo_distribuicao_row[f"{coluna}__preenchidos"],
                resumo_distribuicao_row[f"{coluna}__media"],
                resumo_distribuicao_row[f"{coluna}__desvio_padrao"],
                resumo_distribuicao_row[f"{coluna}__minimo"],
                resumo_distribuicao_row[f"{coluna}__maximo"]
            )
        )

resumo_distribuicao_df = spark.createDataFrame(
    resumo_distribuicao,
    [
        "coluna",
        "preenchidos",
        "media",
        "desvio_padrao",
        "minimo",
        "maximo"
    ]
)

display(resumo_distribuicao_df)

In [0]:
# ============================================================
# 04.6 FREQUÊNCIA DOS VALORES
# ============================================================

# O QUE FAZ:
# Calcula a frequência absoluta e percentual dos valores presentes em cada coluna analisável.

# COMO FAZ:
# Realiza uma agregação por coluna e valor, calculando a quantidade de ocorrências e o percentual de participação de cada valor no total de registros da respectiva coluna.

# POR QUE É IMPORTANTE:
# Permite identificar valores predominantes, categorias raras, possíveis concentrações e comportamentos assimétricos.

# PERGUNTA RESPONDIDA:
# "Com que frequência cada valor aparece em cada coluna?"

frequencias = []

for coluna in colunas_distribution:

    frequencia_coluna = (
        df_prepared
        .groupBy(F.col(coluna).cast("string").alias("valor"))
        .agg(
            F.count("*").alias("frequencia")
        )
        .withColumn(
            "coluna",
            F.lit(coluna)
        )
        .withColumn(
            "pct_valor",
            F.col("frequencia") / F.lit(total_registros) * 100
        )
        .select(
            "coluna",
            "valor",
            "frequencia",
            "pct_valor"
        )
    )

    frequencias.append(frequencia_coluna)

frequencia_valores_df = frequencias[0]

for frequencia in frequencias[1:]:
    frequencia_valores_df = frequencia_valores_df.unionByName(
        frequencia
    )

display(
    frequencia_valores_df.orderBy(
        "coluna",
        F.desc("frequencia")
    )
)

In [0]:
# ============================================================
# 04.7 TOP VALORES POR COLUNA
# ============================================================

# O QUE FAZ:
# Identifica os valores mais frequentes de cada coluna.

# COMO FAZ:
# Utiliza a frequência calculada anteriormente e aplica uma janela particionada por coluna para criar o ranking dos valores mais frequentes.

# POR QUE É IMPORTANTE:
# Facilita a identificação dos valores predominantes sem necessidade de analisar toda a distribuição.

# PERGUNTA RESPONDIDA:
# "Quais são os valores que mais aparecem em cada coluna?"

window_top_valores = (
    Window
    .partitionBy("coluna")
    .orderBy(
        F.desc("frequencia"),
        F.asc("valor")
    )
)

frequencia_top5 = (
    frequencia_valores_df
    .withColumn(
        "ranking",
        F.row_number().over(window_top_valores)
    )
    .filter(
        F.col("ranking") <= TOP_N
    )
    .orderBy(
        "coluna",
        "ranking"
    )
)

display(frequencia_top5)

In [0]:
# ============================================================
# 04.8 CONCENTRAÇÃO DOS VALORES
# ============================================================

# O QUE FAZ:
# Mede quanto da distribuição de cada coluna está concentrado nos valores mais frequentes.

# COMO FAZ:
# Utiliza o ranking dos valores e calcula a participação acumulada dos Top 1, Top 3 e Top 5 valores.

# POR QUE É IMPORTANTE:
# Uma concentração elevada pode indicar baixa diversidade, forte predominância de categorias ou possível desequilíbrio na distribuição.

# PERGUNTA RESPONDIDA:
# "Qual percentual dos registros está concentrado nos principais valores de cada coluna?"

concentracao_base = (
    frequencia_valores_df
    .withColumn(
        "ranking",
        F.row_number().over(window_top_valores)
    )
)

agregacoes_concentracao = []

for n in TOP_CONCENTRATION_N:

    agregacoes_concentracao.append(
        F.sum(
            F.when(
                F.col("ranking") <= n,
                F.col("pct_valor")
            ).otherwise(0)
        ).alias(f"concentracao_top{n}")
    )

concentracao_df = (
    concentracao_base
    .groupBy("coluna")
    .agg(*agregacoes_concentracao)
)

display(
    concentracao_df
    .orderBy(F.desc("concentracao_top1"))
)

In [0]:
# ============================================================
# 04.9 ÍNDICE DE DIVERSIDADE
# ============================================================

# O QUE FAZ:
# Calcula o Índice de Diversidade de Shannon para cada coluna.

# COMO FAZ:
# Utiliza a proporção de ocorrência de cada valor e calcula a entropia de Shannon:
#
# H = - Σ p(x) * log2(p(x))
#
# Quanto maior o valor de H, maior tende a ser a diversidade observada na distribuição.

# POR QUE É IMPORTANTE:
# Permite avaliar a diversidade da distribuição de forma quantitativa, complementando a análise de concentração.

# PERGUNTA RESPONDIDA:
# "Quão diversificados são os valores presentes em cada coluna?"

diversidade_df = (
    frequencia_valores_df
    .groupBy("coluna")
    .agg(
        F.sum(
            -(
                (F.col("frequencia") / F.lit(total_registros))
                *
                F.log2(
                    F.col("frequencia") / F.lit(total_registros)
                )
            )
        ).alias("diversidade_shannon")
    )
)

display(
    diversidade_df.orderBy(
        F.desc("diversidade_shannon")
    )
)

In [0]:
# ============================================================
# 04.10 ÍNDICE DE DOMINÂNCIA
# ============================================================

# O QUE FAZ:
# Calcula o percentual de participação do valor mais frequente de cada coluna.

# COMO FAZ:
# Utiliza a concentração Top 1 já calculada anteriormente.

# POR QUE É IMPORTANTE:
# A dominância mostra de forma direta o quanto o valor mais frequente representa da distribuição da coluna.

# PERGUNTA RESPONDIDA:
# "Qual é a participação do valor mais frequente em cada coluna?"

dominancia_df = (
    concentracao_df
    .select(
        "coluna",
        F.col("concentracao_top1").alias(
            "indice_dominancia"
        )
    )
)

display(
    dominancia_df.orderBy(
        F.desc("indice_dominancia")
    )
)

In [0]:
# ============================================================
# 04.11 REGULARIDADE DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Classifica a regularidade da distribuição dos valores de cada coluna com base na concentração do valor predominante.

# COMO FAZ:
# Utiliza o índice de dominância para estabelecer uma classificação interpretativa.

# POR QUE É IMPORTANTE:
# Permite transformar uma métrica estatística em uma leitura mais simples sobre o comportamento da distribuição.

# PERGUNTA RESPONDIDA:
# "A distribuição da coluna é concentrada ou relativamente distribuída entre seus valores?"

regularidade_df = (
    dominancia_df
    .withColumn(
        "classificacao_regularidade",
        F.when(
            F.col("indice_dominancia") >= 80,
            "ALTAMENTE CONCENTRADA"
        )
        .when(
            F.col("indice_dominancia") >= 50,
            "CONCENTRADA"
        )
        .when(
            F.col("indice_dominancia") >= 20,
            "MODERADAMENTE DISTRIBUÍDA"
        )
        .otherwise(
            "DISTRIBUÍDA"
        )
    )
)

display(
    regularidade_df.orderBy(
        F.desc("indice_dominancia")
    )
)

In [0]:
# ============================================================
# 04.12 PERFIL CONSOLIDADO DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Consolida as principais métricas calculadas pelo Distribution Profile em um único DataFrame.

# COMO FAZ:
# Realiza joins entre frequência, concentração, diversidade, dominância e regularidade.

# POR QUE É IMPORTANTE:
# Cria uma visão única da distribuição de cada coluna, facilitando análises posteriores e a utilização do resultado pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é o perfil completo da distribuição de cada coluna?"

distribution_profile_df = (
    concentracao_df
    .join(
        diversidade_df,
        on="coluna",
        how="left"
    )
    .join(
        regularidade_df,
        on="coluna",
        how="left"
    )
    .orderBy("coluna")
)

display(distribution_profile_df)

In [0]:
# ============================================================
# 04.13 RESUMO EXECUTIVO DO DISTRIBUTION PROFILE
# ============================================================

# O QUE FAZ:
# Cria uma visão executiva das principais métricas de distribuição de cada coluna.

# COMO FAZ:
# Seleciona as métricas mais relevantes do perfil consolidado e cria uma leitura interpretativa da distribuição.

# POR QUE É IMPORTANTE:
# Facilita a interpretação dos resultados sem necessidade de analisar todas as métricas individualmente.

# PERGUNTA RESPONDIDA:
# "Qual é o comportamento predominante da distribuição de cada coluna?"

distribution_summary_df = (
    distribution_profile_df
    .select(
        "coluna",
        "concentracao_top1",
        "concentracao_top3",
        "concentracao_top5",
        "diversidade_shannon",
        "indice_dominancia",
        "classificacao_regularidade"
    )
    .withColumn(
        "leitura",
        F.when(
            F.col("indice_dominancia") >= 80,
            "Distribuição fortemente concentrada."
        )
        .when(
            F.col("indice_dominancia") >= 50,
            "Distribuição concentrada nos principais valores."
        )
        .when(
            F.col("diversidade_shannon") >= 5,
            "Distribuição com elevada diversidade de valores."
        )
        .otherwise(
            "Distribuição relativamente diversificada."
        )
    )
    .orderBy(
        F.desc("indice_dominancia")
    )
)

display(distribution_summary_df)

In [0]:
# ============================================================
# 04.14 VISUALIZAÇÃO — TOP VALORES
# ============================================================

# O QUE FAZ:
# Prepara os dados para visualização dos principais valores de cada coluna.

# COMO FAZ:
# Utiliza o resultado do ranking Top N calculado anteriormente.

# POR QUE É IMPORTANTE:
# Permite visualizar rapidamente quais valores dominam a distribuição de cada atributo.

# PERGUNTA RESPONDIDA:
# "Quais valores predominam visualmente em cada coluna?"

visualizacao_top_valores_df = (
    frequencia_top5
    .select(
        "coluna",
        "valor",
        "frequencia",
        "pct_valor",
        "ranking"
    )
    .orderBy(
        "coluna",
        "ranking"
    )
)

display(visualizacao_top_valores_df)

In [0]:
# ============================================================
# 04.15 VISUALIZAÇÃO — CONCENTRAÇÃO
# ============================================================

# O QUE FAZ:
# Prepara as métricas de concentração para visualização.

# COMO FAZ:
# Seleciona os percentuais acumulados dos Top 1, Top 3 e Top 5 valores de cada coluna.

# POR QUE É IMPORTANTE:
# Permite comparar visualmente o nível de concentração das diferentes colunas.

# PERGUNTA RESPONDIDA:
# "Quais colunas apresentam maior concentração de valores?"

visualizacao_concentracao_df = (
    concentracao_df
    .select(
        "coluna",
        "concentracao_top1",
        "concentracao_top3",
        "concentracao_top5"
    )
    .orderBy(
        F.desc("concentracao_top1")
    )
)

display(visualizacao_concentracao_df)

In [0]:
# ============================================================
# 04.16 VISUALIZAÇÃO — DIVERSIDADE X DOMINÂNCIA
# ============================================================

# O QUE FAZ:
# Prepara uma visão comparativa entre diversidade e dominância das colunas.

# COMO FAZ:
# Utiliza o índice de diversidade de Shannon e o índice de dominância calculados anteriormente.

# POR QUE É IMPORTANTE:
# Permite identificar diferentes comportamentos de distribuição, como colunas com alta diversidade e baixa dominância ou baixa diversidade e alta dominância.

# PERGUNTA RESPONDIDA:
# "Como diversidade e dominância se relacionam entre as colunas?"

visualizacao_diversidade_dominancia_df = (
    distribution_profile_df
    .select(
        "coluna",
        "diversidade_shannon",
        "indice_dominancia",
        "classificacao_regularidade"
    )
    .orderBy(
        F.desc("diversidade_shannon")
    )
)

display(visualizacao_diversidade_dominancia_df)